# Pipeline Failure Intelligence
## Build an AI Agent that Understands Your Data Pipelines

Your pipelines fail. Logs are scattered. Root cause analysis is manual, slow, and happens
after the damage is done. This notebook builds an agent that answers natural language
questions over 90 days of pipeline run history -- combining structured metadata queries
with semantic search over log text, all stored in a single Oracle AI Database instance.

No separate vector store. No extra infrastructure. One database, one query language,
one agent that reasons across both.

## Prerequisites

Before running this notebook, ensure the following are in place:

### 1. Docker Desktop
Docker Desktop must be installed and running. Start the full stack from the project root:
```bash
docker compose up --build
```
Wait until `oracle-26ai` shows as **healthy** in `docker ps` output before running any cells.

### 2. Ollama -- two models required

Ollama must be running locally. Pull both models before starting:

```bash
ollama pull qwen2.5:7b       # LLM for agent reasoning
ollama pull nomic-embed-text # Dedicated embedding model for vector search
```

> **Why a separate embedding model?** Chat models like `qwen2.5:7b` are optimized for generating
> coherent text. When used to produce embeddings, they generate vectors that are less
> semantically precise for similarity search -- two log messages that mean the same thing
> may land far apart in the vector space. `nomic-embed-text` is a purpose-built embedding
> model (~274 MB) trained specifically for semantic similarity tasks. Using it instead of
> the chat model produces meaningfully better vector search results in this demo.

### 3. Alternative: OCI Always Free Autonomous Database

If you have an OCI Always Free account, you can point this notebook at an Autonomous Database
instead of the local Docker container. Just update `DB_HOST`, `DB_PORT`, and `DB_SERVICE`
in the config cell below.

### Docker Setup

Copy the block below into a file named `docker-compose.yml` in the same directory
as this notebook, then start the stack:

```yaml
services:
  oracle-db:
    image: container-registry.oracle.com/database/free:latest
    container_name: oracle-26ai
    environment:
      ORACLE_PWD: Welcome1
    ports:
      - "1521:1521"
    healthcheck:
      test: ["CMD", "healthcheck.sh"]
      interval: 30s
      timeout: 10s
      retries: 5

  jupyter:
    build: ./jupyter
    container_name: jupyter
    ports:
      - "8888:8888"
    volumes:
      - ./notebooks:/home/jovyan/notebooks
    environment:
      - DB_HOST=oracle-db
      - DB_PORT=1521
      - DB_SERVICE=FREEPDB1
      - DB_USER=system
      - DB_PASSWORD=Welcome1
      - OLLAMA_BASE_URL=http://host.docker.internal:11434
      - OLLAMA_MODEL=qwen2.5:7b
      - EMBEDDING_MODEL=nomic-embed-text
    depends_on:
      - oracle-db
```

Start the stack with:
```bash
docker compose up --build
```

Wait until `oracle-26ai` shows as **healthy** before running any cells:

```bash
docker ps | grep oracle-26ai
# oracle-26ai   ...   (healthy)
```
Then open Jupyter in your browser at http://localhost:8888 and open this notebook.
The token is printed in the container logs if prompted
```bash


docker logs jupyter 2>&1 | grep to```
 depends_on:
      - oracle-db
``````


In [1]:
import os

# -- LLM Provider -------------------------------------------------------------
# Default: Ollama (local, free). To use OpenAI-compatible API:
#   set OLLAMA_BASE_URL=https://api.openai.com/v1
#   set OLLAMA_MODEL=gpt-4o
#   set OPENAI_API_KEY=sk-...
#   set EMBEDDING_MODEL=text-embedding-3-small
LLM_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
LLM_MODEL    = os.getenv("OLLAMA_MODEL", "qwen2.5:7b")
LLM_API_KEY  = os.getenv("OPENAI_API_KEY", "ollama")  # ignored by Ollama

# -- Embedding model -----------------------------------------------------------
# Always use a dedicated embedding model -- not the chat model.
# nomic-embed-text is purpose-built for semantic similarity and runs locally via Ollama.
# For OpenAI: use text-embedding-3-small or text-embedding-3-large
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "nomic-embed-text")

# -- Oracle AI Database --------------------------------------------------------
DB_HOST     = os.getenv("DB_HOST", "localhost")
DB_PORT     = os.getenv("DB_PORT", "1521")
DB_SERVICE  = os.getenv("DB_SERVICE", "FREEPDB1")
DB_USER     = os.getenv("DB_USER", "system")
DB_PASSWORD = os.getenv("DB_PASSWORD", "Welcome1")
DB_DSN      = f"{DB_HOST}:{DB_PORT}/{DB_SERVICE}"

# -- Data generator ------------------------------------------------------------
DAYS_OF_HISTORY = int(os.getenv("DAYS_OF_HISTORY", "90"))
RANDOM_SEED     = 42

print(f"LLM:        {LLM_MODEL} @ {LLM_BASE_URL}")
print(f"Embeddings: {EMBEDDING_MODEL}")
print(f"Oracle:     {DB_USER}@{DB_DSN}")
print(f"History:    {DAYS_OF_HISTORY} days")

LLM:        qwen2.5:7b @ http://host.docker.internal:11434
Embeddings: nomic-embed-text
Oracle:     system@oracle-db:1521/FREEPDB1
History:    90 days


## Imports

- **`oracledb`** -- Oracle's thin Python driver, no Oracle Client installation required
- **`langchain_ollama`** -- LLM and embedding wrappers for locally-running Ollama models
- **`langchain_oracledb`** -- Oracle-native vector store (`OracleVS`) and persistent chat history (`OracleChatMessageHistory`)
- **`langchain.agents`** -- ReAct agent that decides which tool to call at each reasoning step
- **`langchain.tools`** -- `@tool` decorator for wrapping Python functions as agent-callable tools

In [2]:
# Copyright (c) 2024 Oracle and/or its affiliates.
# Licensed under the Universal Permissive License v 1.0

import array
import os
import uuid
import random
from datetime import datetime, timedelta

import oracledb
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_oracledb import OracleVS
from langchain_oracledb.vectorstores.oraclevs import DistanceStrategy
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain.agents import create_agent


## Connect to Oracle AI Database

Using `oracledb` in thin mode -- no Oracle Client installation needed. The connection
string follows the standard `host:port/service` format.

In [3]:
try:
    connection = oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=DB_DSN)
    cursor = connection.cursor()
    cursor.execute("SELECT SYSDATE FROM DUAL")
    print(f"✅ Connected to Oracle AI Database: {cursor.fetchone()[0]}")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("→ Check that docker compose up completed and Oracle shows as healthy")
    print("→ Run: docker ps | grep oracle-26ai")
    raise

# SYSTEM user defaults to the SYSTEM tablespace which uses manual segment space
# management (MSSM). Oracle JSON and VECTOR column types require automatic segment
# space management (ASSM). Switch the default tablespace to USERS which has ASSM.
cursor.execute("ALTER USER system DEFAULT TABLESPACE USERS QUOTA UNLIMITED ON USERS")
connection.commit()
print("✅ Default tablespace set to USERS (required for VECTOR/JSON columns)")


✅ Connected to Oracle AI Database: 2026-04-27 08:19:27
✅ Default tablespace set to USERS (required for VECTOR/JSON columns)


---
## Step 2 -- Generate Sample Data (run once)

> **Re-running this section drops and recreates all tables -- safe to run multiple times.**

This demo uses synthetic pipeline run logs for a fictional **NYC Taxi data warehouse**.
The pipelines process NYC Taxi trip data -- a dataset every data engineer recognizes.

The logs reference real NYC Taxi schema columns:
`tpep_pickup_datetime`, `tpep_dropoff_datetime`, `PULocationID`, `DOLocationID`,
`fare_amount`, `passenger_count`, `trip_distance`, `payment_type`, and others.

Approximately **90 days of history** with an **~85% overall success rate** -- realistic
for a production data warehouse. Three tiers of failures are embedded:

| Tier | Type | Discovery method |
|------|------|------------------|
| 1 | Structured failures (error codes, retry storms) | SQL queries |
| 2 | Text-rich failures (schema drift, silent data quality) | Vector similarity search |
| 3 | Cross-run patterns (Monday slowdowns, EOM failures, duration creep) | Agent reasoning |

The generator is fully transparent. Inspect and modify it before running the demo
if you want different failure patterns, DAG counts, or date ranges.

In [4]:
# -----------------------------------------------------------------------------
# Synthetic NYC Taxi pipeline run generator
# Three tiers of failure are embedded in the output -- see inline comments.
# -----------------------------------------------------------------------------

random.seed(RANDOM_SEED)

# DAGs and their tasks -- mirrors a realistic Airflow setup for a taxi DW
DAGS: dict[str, list[str]] = {
    "nyc_taxi_ingestion": [
        "extract_from_s3",
        "validate_schema",
        "load_raw",
    ],
    "fare_aggregation_daily": [
        "compute_fares",
        "apply_surge_multiplier",
        "write_to_warehouse",
    ],
    "borough_partition_load": [
        "partition_by_borough",
        "load_manhattan",
        "load_brooklyn",
        "load_queens",
        "load_bronx",
        "load_staten_island",
    ],
    "tip_analysis_weekly": [
        "join_payment_data",
        "compute_tip_percentages",
        "update_driver_stats",
    ],
    "data_quality_checks": [
        "check_null_rates",
        "check_schema_drift",
        "check_row_counts",
        "check_fare_ranges",
    ],
}

# Base duration ranges (seconds) per task category
DURATION_RANGES: dict[str, tuple[int, int]] = {
    "extract_from_s3":        (180, 600),
    "validate_schema":        (30,  90),
    "load_raw":               (300, 800),
    "compute_fares":          (120, 400),
    "apply_surge_multiplier": (60,  180),
    "write_to_warehouse":     (90,  300),
    "partition_by_borough":   (120, 350),
    "load_manhattan":         (180, 500),
    "load_brooklyn":          (150, 420),
    "load_queens":            (140, 400),
    "load_bronx":             (100, 280),
    "load_staten_island":     (80,  220),
    "join_payment_data":      (90,  260),
    "compute_tip_percentages":(60,  200),
    "update_driver_stats":    (45,  160),
    "check_null_rates":       (30,  90),
    "check_schema_drift":     (25,  80),
    "check_row_counts":       (20,  70),
    "check_fare_ranges":      (25,  75),
}

# Row counts per task category
ROWS_RANGES: dict[str, tuple[int, int]] = {
    "extract_from_s3":        (1_000_000, 4_000_000),
    "load_raw":               (1_000_000, 4_000_000),
    "check_row_counts":       (1_000_000, 4_000_000),
    "check_null_rates":       (1_000_000, 4_000_000),
    "compute_fares":          (50_000,    200_000),
    "apply_surge_multiplier": (50_000,    200_000),
    "write_to_warehouse":     (50_000,    200_000),
    "partition_by_borough":   (50_000,    200_000),
    "load_manhattan":         (20_000,    80_000),
    "load_brooklyn":          (15_000,    60_000),
    "load_queens":            (12_000,    50_000),
    "load_bronx":             (8_000,     30_000),
    "load_staten_island":     (2_000,     8_000),
    "join_payment_data":      (50_000,    200_000),
    "compute_tip_percentages":(50_000,    200_000),
    "update_driver_stats":    (10_000,    40_000),
    "validate_schema":        (0,         0),
    "check_schema_drift":     (0,         0),
    "check_fare_ranges":      (0,         0),
}

# -- Tier 2: realistic S3 parquet path pool -----------------------------------
S3_PARTITIONS = [
    "yellow/2024-01", "yellow/2024-02", "yellow/2024-03",
    "green/2024-01",  "green/2024-02",  "green/2024-03",
    "fhv/2024-01",    "fhv/2024-02",    "fhv/2024-03",
]


def _monday_multiplier(dt: datetime) -> float:
    """Tier 3: Monday runs take 40-65% longer than Tuesday-Friday."""
    return random.uniform(1.40, 1.65) if dt.weekday() == 0 else 1.0


def _weekly_creep_multiplier(dag_id: str, week_num: int) -> float:
    """Tier 3: tip_analysis_weekly grows ~8% per week for 4 weeks, then OOM."""
    if dag_id == "tip_analysis_weekly":
        return 1.0 + (0.08 * min(week_num, 4))
    return 1.0


def _should_fail(dag_id: str, task_id: str, exec_date: datetime) -> bool:
    """Determine whether this run should fail, with tier-specific failure rates."""
    is_eom = exec_date.day >= 28
    if dag_id == "fare_aggregation_daily" and is_eom:
        return random.random() < 0.45   # Tier 3: EOM failures 3x more common
    if task_id == "load_staten_island":
        return random.random() < 0.20   # Tier 1: flaky smaller borough source
    if task_id == "load_raw":
        return random.random() < 0.08   # Tier 1: NULL constraint violations
    return random.random() < 0.06       # baseline 6% failure rate


def _build_log_lines(
    dag_id: str,
    task_id: str,
    exec_date: datetime,
    status: str,
    error_code: str | None,
    rows: int,
    week_num: int,
    retry_count: int,
) -> list[tuple[str, str]]:
    """
    Return a list of (log_level, message) tuples.
    Every run gets 3-8 lines: realistic INFO during execution,
    then ERROR/WARNING at the failure point.
    Tier 2 patterns appear only in text, not in structured fields.
    """
    date_str  = exec_date.strftime("%Y-%m-%d")
    part_idx  = (exec_date.timetuple().tm_yday + hash(dag_id)) % len(S3_PARTITIONS)
    partition = S3_PARTITIONS[part_idx]
    part_num  = random.randint(0, 99)
    lines: list[tuple[str, str]] = []

    # -- Task-specific INFO lines ---------------------------------------------
    if task_id == "extract_from_s3":
        lines.append(("INFO", f"Starting extraction from s3://nyc-taxi-data/{partition}/part-{part_num:05d}.parquet"))
        lines.append(("INFO", "Authenticating with IAM role arn:aws:iam::123456789012:role/NycTaxiDataReader"))
        if retry_count > 0:
            for r in range(retry_count):
                lines.append(("WARNING", f"Retry {r+1}/{retry_count}: S3 connection timeout after 30s, backing off"))
        lines.append(("INFO", f"Downloaded {random.randint(800, 2400)} MB from S3 in {random.randint(45, 180)}s"))

    elif task_id == "validate_schema":
        lines.append(("INFO", f"Validating schema for {partition} dataset against expected 47-column spec"))
        # Tier 2: schema drift lives only in log text -- no error_code in the structured fields
        if dag_id == "nyc_taxi_ingestion" and exec_date.weekday() == 2 and exec_date.day % 7 == 0:
            lines.append((
                "WARNING",
                "Column tpep_dropoff_datetime type changed from TIMESTAMP(6) to VARCHAR2(50) in "
                f"upstream S3 source ({partition}) -- downstream transforms may fail",
            ))
        else:
            lines.append(("INFO", "Schema validation passed for 47 columns, 0 drift detected"))

    elif task_id == "load_raw":
        lines.append(("INFO", "Initiating bulk load into NYC_TAXI_RAW table"))
        lines.append(("INFO", f"Loaded {rows:,} rows into NYC_TAXI_RAW from {partition}"))
        if error_code == "ORA-01400":
            lines.append((
                "ERROR",
                f"ORA-01400: cannot insert NULL into (\"NYC_TAXI_RAW\").(\"TRIP_DISTANCE\") -- "
                f"{random.randint(142, 8900):,} rows affected in partition {partition}/part-{part_num:05d}",
            ))
            if exec_date.weekday() == 0:
                lines.append(("INFO", "Note: NULL rate spike common on Mondays -- upstream provider batch may be incomplete"))

    elif task_id == "compute_fares":
        lines.append(("INFO", f"Computing fare breakdown for {date_str}: {rows:,} trips in scope"))
        lines.append(("INFO", "Applying fare rules: base_fare + distance_charge + congestion_surcharge + mta_tax"))
        if error_code == "ORA-01438":
            lines.append((
                "ERROR",
                f"ORA-01438: value larger than specified precision for column FARE_AMOUNT -- "
                f"detected {random.randint(12, 340):,} records with fare_amount > 9999.99 "
                f"(column precision: NUMBER(6,2)). End-of-month surge pricing may exceed column bounds.",
            ))

    elif task_id == "apply_surge_multiplier":
        lines.append(("INFO", f"Applying surge multiplier for {date_str}: peak hours 07:00-09:00, 17:00-19:30"))
        lines.append(("INFO", f"Surge factor range: 1.2x - 3.4x across {rows:,} eligible trips"))

    elif task_id == "write_to_warehouse":
        lines.append(("INFO", f"Writing {rows:,} aggregated fare records to FARE_AGGREGATES_DAILY"))
        lines.append(("INFO", f"Partition key: execution_date={date_str}"))

    elif task_id == "partition_by_borough":
        lines.append(("INFO", f"Partitioning {rows:,} trip records by PULocationID and DOLocationID"))
        lines.append(("INFO", "Borough mapping: Manhattan(1-109), Brooklyn(110-183), Queens(184-228), Bronx(229-265), Staten Island(266-265)"))

    elif task_id in ("load_manhattan", "load_brooklyn", "load_queens", "load_bronx"):
        borough = task_id.replace("load_", "").title()
        lines.append(("INFO", f"Loading {rows:,} trip records into TRIPS_{borough.upper()} partition"))
        lines.append(("INFO", f"Target table: NYC_TAXI_BOROUGH_FACTS partition {borough.upper()}"))

    elif task_id == "load_staten_island":
        lines.append(("INFO", f"Loading {rows:,} FHV trip records into TRIPS_STATEN_ISLAND"))
        if error_code == "TIMEOUT":
            lines.append((
                "ERROR",
                f"Task exceeded maximum runtime of 3600s -- Staten Island FHV source responded "
                f"after 3614s (SLA: 3600s). Source system may be under maintenance. "
                f"Last checkpoint: {random.randint(1200, 3400):,} of {rows:,} rows committed.",
            ))

    elif task_id == "join_payment_data":
        lines.append(("INFO", f"Joining {rows:,} trip records with PAYMENT_METHODS on payment_type"))
        lines.append(("INFO", "Payment type distribution: Credit(58%), Cash(31%), No-charge(8%), Dispute(3%)"))
        # Tier 3: OOM on week 5 from duration creep
        if dag_id == "tip_analysis_weekly" and week_num >= 5 and status == "failed":
            lines.append((
                "ERROR",
                f"Executor OOM: join_payment_data exceeded 8192MB heap limit after processing "
                f"{random.randint(3_200_000, 4_800_000):,} rows. Week {week_num} dataset has "
                f"grown ~8% per week since week 1 -- consider partitioned join strategy.",
            ))

    elif task_id == "compute_tip_percentages":
        lines.append(("INFO", f"Computing tip_pct = tip_amount / fare_amount for {rows:,} credit card transactions"))
        lines.append(("INFO", "Excluding cash transactions (tip_amount always NULL for payment_type=2)"))

    elif task_id == "update_driver_stats":
        lines.append(("INFO", f"Updating DRIVER_STATS_WEEKLY for {rows:,} unique drivers"))
        lines.append(("INFO", "Metrics: avg_tip_pct, median_trip_distance, trips_per_shift, revenue_per_hour"))

    elif task_id == "check_null_rates":
        null_pct = random.uniform(0.1, 2.5)
        lines.append(("INFO", f"Checking NULL rates across {rows:,} rows for mandatory columns"))
        # Tier 2: passenger_count=0 warning lives only in log text
        if random.random() < 0.18:
            pct = round(random.uniform(5.2, 48.7), 1)
            lines.append((
                "WARNING",
                f"WARNING: {pct}% of Yellow Taxi records for {date_str} have passenger_count = 0, "
                f"exceeding alert threshold of 5%. This may indicate meter-on dispatch records "
                f"or GPS-triggered trip starts without passenger confirmation.",
            ))
        else:
            lines.append(("INFO", f"NULL rate check passed: max null_pct={null_pct:.2f}% across mandatory columns"))

    elif task_id == "check_schema_drift":
        lines.append(("INFO", f"Comparing upstream schema snapshot ({date_str}) against baseline v4.2"))
        lines.append(("INFO", "47 columns checked: 0 type changes, 0 dropped columns, 0 new columns"))

    elif task_id == "check_row_counts":
        lines.append(("INFO", f"Row count validation: NYC_TAXI_RAW={rows:,} rows for {date_str}"))
        # Tier 2: silent dedup lives only in log text
        if random.random() < 0.12:
            dupes = random.randint(120, 4800)
            lines.append((
                "INFO",
                f"Deduplication removed {dupes:,} duplicate trip_ids from FHV dataset ({date_str}) -- "
                f"verify record count with upstream provider. Duplicates may indicate double-dispatch "
                f"in FHV broker system. trip_id range: {random.randint(100000,999999)}-{random.randint(1000000,9999999)}",
            ))
        else:
            lines.append(("INFO", "Row count within expected range +/-2% of 7-day rolling average"))

    elif task_id == "check_fare_ranges":
        lines.append(("INFO", "Validating fare_amount ranges: expected $2.50-$500.00 per TLC regulations"))
        lines.append(("INFO", f"Out-of-range records: {random.randint(0, 12):,} flagged for manual review"))

    # -- Generic completion or failure line -----------------------------------
    if status == "success" and error_code is None:
        lines.append(("INFO", f"Task completed successfully in {random.randint(10, 60)}s post-processing"))
    elif status == "failed" and error_code not in ("ORA-01400", "ORA-01438", "TIMEOUT"):
        lines.append((
            "ERROR",
            f"Unexpected failure in {task_id}: executor exited with code 1. "
            f"Check task logs for {dag_id}/{task_id}/{exec_date.strftime('%Y%m%dT%H%M%S')}",
        ))

    return lines


# -----------------------------------------------------------------------------
# Main generation loop
# -----------------------------------------------------------------------------
jobs: list[dict] = []
logs: list[dict] = []

_tier2_schema_sample: str | None = None
_tier2_dq_sample:     str | None = None
_tier2_dedup_sample:  str | None = None

base_date = datetime.now() - timedelta(days=DAYS_OF_HISTORY)

for day_offset in range(DAYS_OF_HISTORY):
    exec_date = base_date + timedelta(days=day_offset)
    week_num  = day_offset // 7
    run_id    = f"manual__{exec_date.strftime('%Y-%m-%dT%H:%M:%S')}"

    for dag_id, tasks in DAGS.items():
        if dag_id == "tip_analysis_weekly" and exec_date.weekday() != 0:
            continue

        for task_id in tasks:
            job_id = str(uuid.uuid4())

            retry_count = 0
            if task_id == "extract_from_s3" and random.random() < 0.08:
                retry_count = 3

            failed = _should_fail(dag_id, task_id, exec_date)

            error_code: str | None = None
            if failed:
                if task_id == "load_raw":
                    error_code = "ORA-01400"
                elif task_id == "compute_fares" and exec_date.day >= 28:
                    error_code = "ORA-01438"
                elif task_id == "load_staten_island":
                    error_code = "TIMEOUT"
                else:
                    error_code = "TASK_FAILED"

            status = "failed" if failed else "success"

            lo, hi   = DURATION_RANGES.get(task_id, (60, 200))
            base_dur = random.randint(lo, hi)
            duration = int(
                base_dur
                * _monday_multiplier(exec_date)
                * _weekly_creep_multiplier(dag_id, week_num)
            )
            duration += retry_count * random.randint(30, 90)

            start_hour = random.randint(0, 5)
            start_time = exec_date.replace(
                hour=start_hour,
                minute=random.randint(0, 59),
                second=random.randint(0, 59),
            )
            end_time = start_time + timedelta(seconds=duration)

            rlo, rhi  = ROWS_RANGES.get(task_id, (0, 0))
            rows_proc = random.randint(rlo, rhi) if rhi > 0 else 0
            if failed:
                rows_proc = int(rows_proc * random.uniform(0.05, 0.80))

            jobs.append({
                "job_id":           job_id,
                "dag_id":           dag_id,
                "task_id":          task_id,
                "run_id":           run_id,
                "execution_date":   exec_date,
                "start_time":       start_time,
                "end_time":         end_time,
                "duration_seconds": duration,
                "status":           status,
                "retry_count":      retry_count,
                "rows_processed":   rows_proc,
                "error_code":       error_code,
            })

            log_lines = _build_log_lines(
                dag_id, task_id, exec_date, status,
                error_code, rows_proc, week_num, retry_count,
            )

            log_ts = start_time
            for level, message in log_lines:
                log_ts += timedelta(seconds=random.randint(5, 60))
                log_entry = {
                    "log_id":      str(uuid.uuid4()),
                    "job_id":      job_id,
                    "log_level":   level,
                    "log_message": message,
                    "logged_at":   log_ts,
                }
                logs.append(log_entry)

                if _tier2_schema_sample is None and "type changed from TIMESTAMP" in message:
                    _tier2_schema_sample = message
                if _tier2_dq_sample is None and "passenger_count = 0" in message:
                    _tier2_dq_sample = message
                if _tier2_dedup_sample is None and "Deduplication removed" in message:
                    _tier2_dedup_sample = message


# -----------------------------------------------------------------------------
# Quality-gate summary -- review before running the rest of the notebook
# -----------------------------------------------------------------------------
from collections import defaultdict

dag_counts:   dict[str, int] = defaultdict(int)
dag_failures: dict[str, int] = defaultdict(int)

for j in jobs:
    dag_counts[j["dag_id"]] += 1
    if j["status"] == "failed":
        dag_failures[j["dag_id"]] += 1

total_jobs   = len(jobs)
total_failed = sum(1 for j in jobs if j["status"] == "failed")
total_logs   = len(logs)
overall_rate = total_failed / total_jobs * 100

print("=" * 68)
print(" DATA GENERATOR SUMMARY -- review before continuing")
print("=" * 68)
print(f"  Total job runs  : {total_jobs:,}")
print(f"  Total log lines : {total_logs:,}")
print(f"  Overall failure : {overall_rate:.1f}%")
print()
print(f"  {'DAG':<35} {'Runs':>6} {'Failures':>9} {'Fail%':>7}")
print(f"  {'-'*35} {'-'*6} {'-'*9} {'-'*7}")
for dag_id in DAGS:
    runs  = dag_counts[dag_id]
    fails = dag_failures[dag_id]
    pct   = fails / runs * 100 if runs else 0.0
    print(f"  {dag_id:<35} {runs:>6,} {fails:>9,} {pct:>6.1f}%")

print()
print("-" * 68)
print(" TIER 2 LOG SAMPLES (discoverable only via vector search)")
print("-" * 68)

tier2_samples = [
    ("Schema drift", _tier2_schema_sample),
    ("DQ warning  ", _tier2_dq_sample),
    ("Silent dedup", _tier2_dedup_sample),
]
for label, sample in tier2_samples:
    if sample:
        words = sample.split()
        line, preview_lines = "", []
        for w in words:
            if len(line) + len(w) + 1 > 60:
                preview_lines.append(line)
                line = w
            else:
                line = (line + " " + w).strip()
        if line:
            preview_lines.append(line)
        indent = " " * 16
        print(f"  {label}: {preview_lines[0]}")
        for pl in preview_lines[1:]:
            print(f"{indent}{pl}")
        print()
    else:
        print(f"  {label}: [not generated with this seed -- try RANDOM_SEED=43]\n")

print("=" * 68)
print(" If the above looks correct, run the next cell to load data into Oracle.")
print("=" * 68)

 DATA GENERATOR SUMMARY -- review before continuing
  Total job runs  : 1,476
  Total log lines : 4,440
  Overall failure : 8.4%

  DAG                                   Runs  Failures   Fail%
  ----------------------------------- ------ --------- -------
  nyc_taxi_ingestion                     270        25    9.3%
  fare_aggregation_daily                 270        25    9.3%
  borough_partition_load                 540        53    9.8%
  tip_analysis_weekly                     36         1    2.8%
  data_quality_checks                    360        20    5.6%

--------------------------------------------------------------------
 TIER 2 LOG SAMPLES (discoverable only via vector search)
--------------------------------------------------------------------
  Schema drift: Column tpep_dropoff_datetime type changed from TIMESTAMP(6)
                to VARCHAR2(50) in upstream S3 source (yellow/2024-01) --
                downstream transforms may fail

  DQ warning  : WARNING: 29.6% of 

In [5]:
# -- Error code examples -------------------------------------------------------
print("-" * 68)
print(" ERROR CODE EXAMPLES")
print("-" * 68)

error_code_to_log: dict = {}
job_error_map = {j["job_id"]: j for j in jobs if j["error_code"]}
for lg in logs:
    job = job_error_map.get(lg["job_id"])
    if job and lg["log_level"] == "ERROR":
        ec = job["error_code"]
        if ec not in error_code_to_log:
            error_code_to_log[ec] = (job["dag_id"], job["task_id"], lg["log_message"])

for ec in sorted(error_code_to_log):
    dag_id, task_id, msg = error_code_to_log[ec]
    words = msg.split()
    line, wrapped = "", []
    for w in words:
        if len(line) + len(w) + 1 > 60:
            wrapped.append(line)
            line = w
        else:
            line = (line + " " + w).strip()
    if line:
        wrapped.append(line)
    indent = " " * 16
    print(f"  {ec:<14}  {wrapped[0]}")
    for wl in wrapped[1:]:
        print(f"{indent}{wl}")
    print(f"  {'':14}  → {dag_id}.{task_id}")
    print()

# -- Data time range -----------------------------------------------------------
print("-" * 68)
print(" DATA TIME RANGE")
print("-" * 68)

ts_min = min(j["start_time"] for j in jobs)
ts_max = max(j["end_time"]   for j in jobs)

print(f"  From : {ts_min.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  To   : {ts_max.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Span : {(ts_max - ts_min).days} days")
print("=" * 68)

--------------------------------------------------------------------
 ERROR CODE EXAMPLES
--------------------------------------------------------------------
  ORA-01400       ORA-01400: cannot insert NULL into
                ("NYC_TAXI_RAW").("TRIP_DISTANCE") -- 1,225 rows affected in
                partition yellow/2024-02/part-00097
                  → nyc_taxi_ingestion.load_raw

  ORA-01438       ORA-01438: value larger than specified precision for column
                FARE_AMOUNT -- detected 49 records with fare_amount >
                9999.99 (column precision: NUMBER(6,2)). End-of-month surge
                pricing may exceed column bounds.
                  → fare_aggregation_daily.compute_fares

  TASK_FAILED     Unexpected failure in extract_from_s3: executor exited with
                code 1. Check task logs for
                nyc_taxi_ingestion/extract_from_s3/20260127T081930
                  → nyc_taxi_ingestion.extract_from_s3

  TIMEOUT         Task exceeded m

## Create Oracle Tables and Load Data

Two tables:
- **`PIPELINE_JOBS`** -- structured run metadata: DAG, task, status, duration, error codes
- **`PIPELINE_LOGS`** -- free-text log lines linked back to each job

The `DROP TABLE IF EXISTS` guard makes this cell safe to re-run. `executemany` is used
for bulk inserts -- inserting thousands of rows one at a time would be order-of-magnitude
slower with a remote database.

In [6]:
cursor = connection.cursor()

# -- Drop existing tables so the notebook is safely re-runnable ---------------
for tbl in ("PIPELINE_LOGS", "PIPELINE_JOBS", "PIPELINE_LOG_VECTORS", "AGENT_MEMORY"):
    try:
        cursor.execute(f"DROP TABLE {tbl} CASCADE CONSTRAINTS")
        print(f"  Dropped {tbl}")
    except oracledb.DatabaseError:
        pass  # table did not exist yet

# -- PIPELINE_JOBS -------------------------------------------------------------
cursor.execute("""
    CREATE TABLE PIPELINE_JOBS (
        job_id           VARCHAR2(64)   PRIMARY KEY,
        dag_id           VARCHAR2(128),
        task_id          VARCHAR2(128),
        run_id           VARCHAR2(128),
        execution_date   DATE,
        start_time       TIMESTAMP,
        end_time         TIMESTAMP,
        duration_seconds NUMBER,
        status           VARCHAR2(16),
        retry_count      NUMBER DEFAULT 0,
        rows_processed   NUMBER,
        error_code       VARCHAR2(64)
    )
""")

# -- PIPELINE_LOGS -------------------------------------------------------------
cursor.execute("""
    CREATE TABLE PIPELINE_LOGS (
        log_id       VARCHAR2(64)   PRIMARY KEY,
        job_id       VARCHAR2(64)   REFERENCES PIPELINE_JOBS(job_id),
        log_level    VARCHAR2(8),
        log_message  VARCHAR2(4000),
        logged_at    TIMESTAMP
    )
""")

# -- Bulk insert jobs ----------------------------------------------------------
job_rows = [
    (
        j["job_id"], j["dag_id"], j["task_id"], j["run_id"],
        j["execution_date"], j["start_time"], j["end_time"],
        j["duration_seconds"], j["status"], j["retry_count"],
        j["rows_processed"], j["error_code"],
    )
    for j in jobs
]
cursor.executemany(
    """
    INSERT INTO PIPELINE_JOBS
        (job_id, dag_id, task_id, run_id, execution_date, start_time,
         end_time, duration_seconds, status, retry_count, rows_processed, error_code)
    VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9, :10, :11, :12)
    """,
    job_rows,
)

# -- Bulk insert logs ----------------------------------------------------------
log_rows = [
    (lg["log_id"], lg["job_id"], lg["log_level"], lg["log_message"], lg["logged_at"])
    for lg in logs
]
cursor.executemany(
    """
    INSERT INTO PIPELINE_LOGS (log_id, job_id, log_level, log_message, logged_at)
    VALUES (:1, :2, :3, :4, :5)
    """,
    log_rows,
)

connection.commit()

# -- Confirm -------------------------------------------------------------------
cursor.execute("SELECT COUNT(*) FROM PIPELINE_JOBS")
job_count = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM PIPELINE_LOGS")
log_count = cursor.fetchone()[0]

print(f"\u2705 PIPELINE_JOBS  : {job_count:,} rows")
print(f"\u2705 PIPELINE_LOGS  : {log_count:,} rows")

  Dropped PIPELINE_LOGS
  Dropped PIPELINE_JOBS
  Dropped PIPELINE_LOG_VECTORS
✅ PIPELINE_JOBS  : 1,476 rows
✅ PIPELINE_LOGS  : 4,440 rows


In [7]:
# -- Sample row without embedding ---------------------------------------------
# Pick the first log entry so the with-embedding cell below can reference it.
sample_log_id = logs[0]["log_id"]
sample_job_id = logs[0]["job_id"]

cursor.execute("SELECT * FROM PIPELINE_JOBS WHERE job_id = :1", [sample_job_id])
job_cols = [d[0] for d in cursor.description]
job_row  = cursor.fetchone()

cursor.execute("SELECT * FROM PIPELINE_LOGS WHERE log_id = :1", [sample_log_id])
log_cols = [d[0] for d in cursor.description]
log_row  = cursor.fetchone()

print("=" * 68)
print(" SAMPLE ROW -- without embedding")
print("=" * 68)
print(" PIPELINE_JOBS")
print("-" * 68)
for col, val in zip(job_cols, job_row):
    print(f"  {col:<20}  {val}")
print()
print(" PIPELINE_LOGS  (first log line for this job)")
print("-" * 68)
for col, val in zip(log_cols, log_row):
    print(f"  {col:<20}  {val}")
print("=" * 68)

 SAMPLE ROW -- without embedding
 PIPELINE_JOBS
--------------------------------------------------------------------
  JOB_ID                4c03c5db-0a1f-44b5-8eb4-36704fd17fd9
  DAG_ID                nyc_taxi_ingestion
  TASK_ID               extract_from_s3
  RUN_ID                manual__2026-01-27T08:19:30
  EXECUTION_DATE        2026-01-27 08:19:30
  START_TIME            2026-01-27 01:08:47
  END_TIME              2026-01-27 01:14:07
  DURATION_SECONDS      320
  STATUS                failed
  RETRY_COUNT           0
  ROWS_PROCESSED        797201
  ERROR_CODE            TASK_FAILED

 PIPELINE_LOGS  (first log line for this job)
--------------------------------------------------------------------
  LOG_ID                8c3d9706-b448-414a-86b9-562fec6c5452
  JOB_ID                4c03c5db-0a1f-44b5-8eb4-36704fd17fd9
  LOG_LEVEL             INFO
  LOG_MESSAGE           Starting extraction from s3://nyc-taxi-data/fhv/2024-03/part-00069.parquet
  LOGGED_AT             2026-01-27 01

## Why One Database for Everything

Traditional observability stacks split structured metadata (relational DB) from
unstructured log text (Elasticsearch, OpenSearch, or a dedicated vector store).
That means two systems to operate, two schemas to keep in sync, two query languages
to write, and a constant risk of the two systems drifting out of agreement.

Oracle AI Database stores both in the same tables, uses the same SQL for both,
and can filter and rank across both in a single query using `VECTOR_DISTANCE()`
alongside standard `WHERE` clauses:

```sql
SELECT j.dag_id, j.execution_date, l.log_message,
       VECTOR_DISTANCE(v.embedding, :query_vec, COSINE) AS similarity
FROM   PIPELINE_LOG_VECTORS v
JOIN   PIPELINE_LOGS l  ON l.log_id = v.id
JOIN   PIPELINE_JOBS j  ON j.job_id = l.job_id
WHERE  j.dag_id = 'nyc_taxi_ingestion'
ORDER  BY similarity
FETCH  FIRST 10 ROWS ONLY
```

No ETL between systems. No vector store to provision. One `connection`, one `cursor`,
one place to look when something goes wrong. This is the architectural point the
agent below is built to demonstrate.

## Generate Embeddings and Store Vectors

`OracleVS` handles embedding storage transparently -- it calls the embedding model,
receives the vectors, and writes them into Oracle using native `VECTOR` column type.
The vectors live alongside the relational data, queryable via `VECTOR_DISTANCE()`.

**This cell takes several minutes** with a local Ollama model. `nomic-embed-text`
processes roughly 50-100 texts per minute on typical hardware. Progress is printed
every 200 rows.

In [8]:
# -- Dedicated embedding model -- not the chat model ---------------------------
# nomic-embed-text is purpose-built for semantic similarity and runs locally via Ollama.
try:
    embeddings = OllamaEmbeddings(
        model=EMBEDDING_MODEL,
        base_url=LLM_BASE_URL,
    )
    # Quick smoke test before embedding thousands of rows
    _ = embeddings.embed_query("connection test")
    print(f"\u2705 Embedding model ready: {EMBEDDING_MODEL} @ {LLM_BASE_URL}")
except Exception as e:
    print(f"\u274c Embedding model unavailable: {e}")
    print(f"\u2192 Ensure Ollama is running and the model is pulled:")
    print(f"    ollama pull {EMBEDDING_MODEL}")
    raise

# -- Fetch all log messages ----------------------------------------------------
cursor.execute("SELECT log_id, log_message FROM PIPELINE_LOGS ORDER BY log_id")
all_log_rows = cursor.fetchall()
log_ids      = [r[0] for r in all_log_rows]
log_texts    = [r[1] for r in all_log_rows]

print(f"  Embedding {len(log_texts):,} log messages with {EMBEDDING_MODEL}...")
print(f"  This will take several minutes on local hardware. Progress below.")
print()

# -- Batch embed and store -- print progress every 200 rows --------------------
BATCH_SIZE = 200

for batch_start in range(0, len(log_texts), BATCH_SIZE):
    batch_end   = min(batch_start + BATCH_SIZE, len(log_texts))
    batch_texts = log_texts[batch_start:batch_end]
    batch_ids   = log_ids[batch_start:batch_end]

    if batch_start == 0:
        # First batch: create the table
        vector_store = OracleVS.from_texts(
            texts=batch_texts,
            embedding=embeddings,
            client=connection,
            table_name="PIPELINE_LOG_VECTORS",
            distance_strategy=DistanceStrategy.COSINE,
            ids=batch_ids,
        )
    else:
        # Subsequent batches: add to existing table
        vector_store.add_texts(texts=batch_texts, ids=batch_ids)

    print(f"  [{batch_end:>5,} / {len(log_texts):,}] embedded and stored")

print()
print(f"\u2705 Stored {len(log_texts):,} log embeddings in Oracle AI Database")
print(f"   Embedding model : {EMBEDDING_MODEL}")
print(f"   Table           : PIPELINE_LOG_VECTORS")

✅ Embedding model ready: nomic-embed-text @ http://host.docker.internal:11434
  Embedding 4,440 log messages with nomic-embed-text...
  This will take several minutes on local hardware. Progress below.

  [  200 / 4,440] embedded and stored
  [  400 / 4,440] embedded and stored
  [  600 / 4,440] embedded and stored
  [  800 / 4,440] embedded and stored
  [1,000 / 4,440] embedded and stored
  [1,200 / 4,440] embedded and stored
  [1,400 / 4,440] embedded and stored
  [1,600 / 4,440] embedded and stored
  [1,800 / 4,440] embedded and stored
  [2,000 / 4,440] embedded and stored
  [2,200 / 4,440] embedded and stored
  [2,400 / 4,440] embedded and stored
  [2,600 / 4,440] embedded and stored
  [2,800 / 4,440] embedded and stored
  [3,000 / 4,440] embedded and stored
  [3,200 / 4,440] embedded and stored
  [3,400 / 4,440] embedded and stored
  [3,600 / 4,440] embedded and stored
  [3,800 / 4,440] embedded and stored
  [4,000 / 4,440] embedded and stored
  [4,200 / 4,440] embedded and stored

In [9]:
# -- Same row with embedding --------------------------------------------------
# Look up the exact same log_id as the cell above (sample_log_id set there).
# OracleVS stores our log_ids in METADATA under '__orcl_internal_doc_id'.
cursor.execute("""
    SELECT l.log_id, l.log_level, l.log_message, l.logged_at, v.text, v.metadata, v.embedding
    FROM   PIPELINE_LOG_VECTORS v
    JOIN   PIPELINE_LOGS l ON JSON_VALUE(v.metadata, '$.__orcl_internal_doc_id') = l.log_id
    WHERE  l.log_id = :1
""", [sample_log_id])
joined_cols = [d[0] for d in cursor.description]
joined_row  = cursor.fetchone()

print("=" * 68)
print(" SAMPLE ROW -- with embedding")
print("=" * 68)

if joined_row:
    for col, val in zip(joined_cols, joined_row):
        if isinstance(val, list):
            dims    = len(val)
            preview = [f"{v:.6f}" for v in val[:6]]
            print(f"  {col:<20}  [{', '.join(preview)}, ...]  ({dims} dimensions)")
        elif isinstance(val, bytes):
            print(f"  {col:<20}  {val.hex()}")
        else:
            print(f"  {col:<20}  {val}")
else:
    print("  (no row found for sample_log_id -- verify embeddings cell ran successfully)")
print("=" * 68)


 SAMPLE ROW -- with embedding
  LOG_ID                8c3d9706-b448-414a-86b9-562fec6c5452
  LOG_LEVEL             INFO
  LOG_MESSAGE           Starting extraction from s3://nyc-taxi-data/fhv/2024-03/part-00069.parquet
  LOGGED_AT             2026-01-27 01:08:54
  TEXT                  Starting extraction from s3://nyc-taxi-data/fhv/2024-03/part-00069.parquet
  METADATA              {'__orcl_internal_doc_id': '8c3d9706-b448-414a-86b9-562fec6c5452'}
  EMBEDDING             array('f', [0.040450237691402435, 0.015210443176329136, -0.17485958337783813, -0.05316894128918648, 0.028915246948599815, 0.01986798830330372, -0.02749624475836754, 0.025041405111551285, -0.0033510071225464344, 0.011186936870217323, -0.08683187514543533, -0.05168701335787773, 0.054099343717098236, -0.044956907629966736, -0.029006754979491234, -0.04265683516860008, -0.04514061659574509, -0.07104741036891937, -0.021012812852859497, 0.05304356664419174, -0.0013424535281956196, -0.03622044622898102, -0.03508946672081947, 

---
## Step 3 -- Build the Agent

Setup is complete. Oracle now holds:
- **`PIPELINE_JOBS`** -- structured run metadata (DAG, task, status, duration, error codes)
- **`PIPELINE_LOGS`** -- free-text log lines linked to each job
- **`PIPELINE_LOG_VECTORS`** -- vector embeddings of every log message, stored natively in Oracle

The next cells build the reasoning agent that queries all three using natural language.

## LangChain Setup: LLM and Agent Tools

Three tools give the agent complementary access to the data:

| Tool | What it does | When the agent uses it |
|------|-------------|------------------------|
| `pipeline_sql_tool` | Executes read-only SQL against `PIPELINE_JOBS` | Counts, durations, error codes, date ranges |
| `log_search_tool` | Semantic search over `PIPELINE_LOG_VECTORS` | Questions about log text, errors, schema changes |
| `hybrid_pipeline_search` | SQL filter + `VECTOR_DISTANCE()` in one query | Questions that combine DAG/date filter with log content |

The `hybrid_pipeline_search` tool is the key differentiator -- it demonstrates Oracle's
converged query capability: structured and vector search in a single SQL statement,
with no application-level join or data movement.

In [10]:
# -- LLM -----------------------------------------------------------------------
try:
    llm = ChatOllama(model=LLM_MODEL, base_url=LLM_BASE_URL, temperature=0)
    _ = llm.invoke("Reply with one word: ready")
    print(f"✅ LLM ready: {LLM_MODEL} @ {LLM_BASE_URL}")
except Exception as e:
    print(f"❌ LLM unavailable: {e}")
    print(f"→ Ensure Ollama is running and the model is pulled:")
    print(f"    ollama pull {LLM_MODEL}")
    raise


# -- Tool 1: Structured SQL ----------------------------------------------------
@tool
def pipeline_sql_tool(sql: str) -> str:
    """
    Execute a read-only SQL SELECT query against the PIPELINE_JOBS table.
    Use this for structured questions: failure counts, durations, error codes,
    retry storms, date ranges, and per-DAG or per-task statistics.
    Returns results as a formatted string. Only SELECT statements are allowed.

    Available columns in PIPELINE_JOBS:
      job_id, dag_id, task_id, run_id, execution_date, start_time, end_time,
      duration_seconds, status, retry_count, rows_processed, error_code

    IMPORTANT -- this is Oracle SQL, not MySQL or PostgreSQL. Date rules:
      - Current date/time : SYSDATE
      - Last 30 days      : WHERE execution_date >= SYSDATE - 30
      - Day of week       : TO_CHAR(execution_date, 'DY')  -- returns MON, TUE, etc.
      - Week number       : TO_CHAR(execution_date, 'IW')
      Do NOT use DATE_SUB(), CURRENT_DATE(), DATEADD(), or any non-Oracle syntax.
      - Failure rate  : AVG(CASE WHEN status = 'failed' THEN 1.0 ELSE 0.0 END) * 100
        (Do NOT use SUM/COUNT for rates -- Oracle integer division truncates to 0)
    """
    print(f"[TOOL] pipeline_sql_tool called with: {sql[:80].strip()}")
    sql_upper = sql.strip().upper()
    if not sql_upper.startswith("SELECT"):
        return "ERROR: Only SELECT statements are permitted."
    for keyword in ("INSERT", "UPDATE", "DELETE", "DROP", "TRUNCATE", "ALTER"):
        if keyword in sql_upper:
            return f"ERROR: {keyword} statements are not permitted."
    try:
        cur = connection.cursor()
        cur.execute(sql)
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
        if not rows:
            return "Query returned 0 rows."
        col_widths = [max(len(c), max((len(str(r[i])) for r in rows), default=0)) for i, c in enumerate(cols)]
        header = "  ".join(c.ljust(col_widths[i]) for i, c in enumerate(cols))
        sep    = "  ".join("-" * w for w in col_widths)
        lines  = [header, sep] + [
            "  ".join(str(v).ljust(col_widths[i]) for i, v in enumerate(row))
            for row in rows
        ]
        return f"{len(rows)} row(s):\n" + "\n".join(lines)
    except Exception as e:
        return f"SQL error: {e}"


# -- Tool 2: Semantic log search -----------------------------------------------
@tool
def log_search_tool(query: str) -> str:
    """
    Semantic similarity search over pipeline log messages.
    Use this for questions that require understanding log text: schema changes,
    error descriptions, data quality warnings, or silent issues that did not
    cause a job failure (status=success but log contains WARNING or INFO about issues).
    Returns the top 5 most semantically relevant log entries with job context.
    """
    print(f"[TOOL] log_search_tool called with: {query}")
    try:
        results = vector_store.similarity_search(query, k=5)
        if not results:
            return "No relevant log messages found."

        # OracleVS stores our log_id in metadata under '__orcl_internal_doc_id',
        # not in the doc.id field (which is OracleVS's internal RAW primary key).
        output_lines: list[str] = []
        for i, doc in enumerate(results, 1):
            log_id = doc.metadata.get("__orcl_internal_doc_id", "")
            cur = connection.cursor()
            cur.execute(
                """
                SELECT j.dag_id, j.task_id, j.execution_date, j.status,
                       l.log_level, l.log_message
                FROM   PIPELINE_LOGS l
                JOIN   PIPELINE_JOBS j ON j.job_id = l.job_id
                WHERE  l.log_id = :1
                """,
                [log_id],
            )
            row = cur.fetchone()
            if row:
                dag_id, task_id, exec_date, status, log_level, log_message = row
                output_lines.append(
                    f"[{i}] {log_level} | {dag_id}.{task_id} | {exec_date} | status={status}\n"
                    f"    {log_message}"
                )
            else:
                output_lines.append(f"[{i}] {doc.page_content}")

        return "\n\n".join(output_lines)
    except Exception as e:
        return f"Search error: {e}"


# -- Tool 3: Hybrid structured + vector search ---------------------------------
@tool
def hybrid_pipeline_search(query: str, dag_id: str = "", date_from: str = "", date_to: str = "") -> str:
    """
    Combined structured filter + semantic vector search in a single Oracle SQL query.
    This is Oracle's converged query capability: filter by DAG and/or date range,
    then rank results by semantic similarity to the query text -- all in one statement.
    Use this when a question has both a structural constraint (specific DAG, time window)
    AND requires understanding log content.

    Parameters:
      query     : natural language description of what you're looking for (required)
      dag_id    : filter to a specific DAG name (optional, leave empty for all DAGs)
      date_from : start date filter in YYYY-MM-DD format (optional)
      date_to   : end date filter in YYYY-MM-DD format (optional)

    Returns top 10 log entries ranked by semantic relevance within the filter.
    """
    print(f"[TOOL] hybrid_pipeline_search called | query={query} | dag={dag_id} | from={date_from} | to={date_to}")
    try:
        raw_embedding = embeddings.embed_query(query)
        query_embedding = array.array('f', raw_embedding)

        where_clauses = []
        bind_params: list = [query_embedding]

        if dag_id:
            where_clauses.append("j.dag_id = :2")
            bind_params.append(dag_id)
        if date_from:
            bind_params.append(date_from)
            where_clauses.append(f"j.execution_date >= TO_DATE(:{len(bind_params)}, 'YYYY-MM-DD')")
        if date_to:
            bind_params.append(date_to)
            where_clauses.append(f"j.execution_date <= TO_DATE(:{len(bind_params)}, 'YYYY-MM-DD')")

        where_sql = ("WHERE " + " AND ".join(where_clauses)) if where_clauses else ""

        # OracleVS stores our log_id in the METADATA JSON under '__orcl_internal_doc_id'.
        sql = f"""
            SELECT j.dag_id, j.task_id, j.execution_date, j.status,
                   l.log_level, l.log_message,
                   VECTOR_DISTANCE(v.embedding, :1, COSINE) AS similarity
            FROM   PIPELINE_LOG_VECTORS v
            JOIN   PIPELINE_LOGS l  ON JSON_VALUE(v.metadata, '$.__orcl_internal_doc_id') = l.log_id
            JOIN   PIPELINE_JOBS j  ON j.job_id = l.job_id
            {where_sql}
            ORDER  BY similarity
            FETCH  FIRST 10 ROWS ONLY
        """

        cur = connection.cursor()
        cur.execute(sql, bind_params)
        rows = cur.fetchall()

        if not rows:
            return "No matching log entries found."

        output_lines: list[str] = []
        for i, (dag, task, exec_date, status, level, message, sim) in enumerate(rows, 1):
            output_lines.append(
                f"[{i}] sim={sim:.4f} | {level} | {dag}.{task} | {exec_date} | status={status}\n"
                f"    {message}"
            )
        return "\n\n".join(output_lines)
    except Exception as e:
        return f"Hybrid search error: {e}"


tools = [pipeline_sql_tool, log_search_tool, hybrid_pipeline_search]
print(f"✅ {len(tools)} tools registered: {', '.join(t.name for t in tools)}")

✅ LLM ready: qwen2.5:7b @ http://host.docker.internal:11434
✅ 3 tools registered: pipeline_sql_tool, log_search_tool, hybrid_pipeline_search


## Build the Agent

A ReAct agent (Reason + Act) loops through: observe the question, decide which tool
to call and with what input, observe the tool's output, decide whether to call
another tool or answer. `max_iterations=8` allows multi-tool reasoning chains.

`InMemoryChatMessageHistory` preserves conversation context in memory --
the agent remembers answers from earlier questions in the session. This means
the final reliability report (question 7) can reference patterns discovered in
questions 1-6 without re-running those queries.

In [11]:
# -- System prompt --
SYSTEM_PROMPT = (
    "You are a data pipeline reliability expert with access to 90 days "
    "of pipeline run history for a NYC Taxi data warehouse. "
    "You have three tools: "
    "(1) pipeline_sql_tool for structured questions -- counts, durations, failure rates, date ranges; "
    "(2) log_search_tool for questions requiring understanding of log text, error messages, "
    "warnings, schema changes, and data quality issues; "
    "(3) hybrid_pipeline_search for questions that combine both -- filter by DAG or date range "
    "AND search log content semantically. "
    "Always consider using multiple tools when a question could benefit from both structured "
    "and semantic perspectives. Be specific: include DAG names, dates, row counts, error codes, "
    "and durations. When you identify patterns, explain them and suggest what to investigate next."
)

# -- LangGraph ReAct agent --
# ChatOllama (not OllamaLLM) is required -- chat models support bind_tools.
agent_executor = create_agent(
    model=llm,
    tools=tools,
)

# -- In-memory chat history --
chat_history = InMemoryChatMessageHistory()


def ask(question: str) -> str:
    """Run one question through the agent, preserving chat history across calls."""
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + list(chat_history.messages) + [HumanMessage(content=question)]
    result = agent_executor.invoke({"messages": messages})
    answer = result["messages"][-1].content
    chat_history.add_user_message(question)
    chat_history.add_ai_message(answer)
    return answer


print("✅ Agent ready (LangGraph ReAct). Chat history active for this session.")


✅ Agent ready (LangGraph ReAct). Chat history active for this session.


## The Demo

Seven questions that build a coherent investigation — starting with SQL to identify the
problem DAG, then drilling in from multiple angles, then directly demonstrating Oracle's
converged query capability, and finally synthesizing everything into a report that
genuinely benefits from the memory of prior answers:

| # | Tool | What it exercises |
|---|------|-------------------|
| 1 | `pipeline_sql_tool` | Tier 1: failure counts by DAG — identifies `borough_partition_load` |
| 2 | `log_search_tool` | Tier 2: error patterns in that DAG's logs — semantic search over log text |
| 3 | `pipeline_sql_tool` | Tier 3: is it getting worse? failure trend by week |
| 4 | `log_search_tool` | Tier 2: are data quality checks also failing? schema drift log search |
| 5 | `pipeline_sql_tool` | Tier 3: day-of-week performance breakdown — Friday as failure outlier |
| 6 | `hybrid_pipeline_search` (direct) | DAG filter + `VECTOR_DISTANCE()` in one Oracle SQL — no agent |
| 7 | memory from Q1–Q6 | Synthesis: full reliability report, no new queries needed |

Watch the `[TOOL]` lines printed before each answer — they show exactly which tool
the agent picked and what it passed in.

In [15]:
# Question 1/6  [Tier 1 -- pipeline_sql_tool]
Q1 = "How many pipeline failures occurred in the last 30 days, and which DAG failed most often? use pipeline_sql_tool."
print("=" * 70)
print(" Q1/6  [Tier 1 -- pipeline_sql_tool]")
print("=" * 70)
print(f" {Q1}")
print("-" * 70)
answer = ask(Q1)
print()
print(answer)


 Q1/6  [Tier 1 -- pipeline_sql_tool]
 How many pipeline failures occurred in the last 30 days, and which DAG failed most often? use pipeline_sql_tool
----------------------------------------------------------------------
[TOOL] pipeline_sql_tool called with: SELECT dag_id, COUNT(*) AS failure_count FROM PIPELINE_JOBS WHERE status = 'fail

In the last 30 days, there were a total of 40 pipeline failures across different DAGs. The DAG that failed most often was `borough_partition_load` with 16 failures, followed by `fare_aggregation_daily` with 12 failures.

Next steps:
- Investigate the `borough_partition_load` and `fare_aggregation_daily` DAGs to understand why they are failing more frequently.
- Check if there are common issues or patterns in these DAGs that could be addressed.
- Monitor their performance closely for any recurring problems.


In [16]:
# Question 2/6  [Tier 2 -- log_search_tool]
Q2 = "What do the error logs say about why borough_partition_load is failing? Search the log text to identify the error patterns."
print("=" * 70)
print(" Q2/6  [Tier 2 -- log_search_tool]")
print("=" * 70)
print(f" {Q2}")
print("-" * 70)
answer = ask(Q2)
print()
print(answer)

 Q2/6  [Tier 2 -- log_search_tool]
 What do the error logs say about why borough_partition_load is failing? Search the log text to identify the error patterns.
----------------------------------------------------------------------
[TOOL] log_search_tool called with: borough_partition_load failure

The error logs indicate that the `partition_by_borough` task within the `borough_partition_load` DAG is failing with an executor exit code of 1. Specifically, this issue has occurred multiple times:

- On April 12, 2026
- February 24, 2026
- April 19, 2026
- February 5, 2026
- March 31, 2026

The error message "executor exited with code 1" suggests that there might be an issue related to the execution environment or a specific task within this partitioning process. To investigate further:

1. **Check Task Logs**: Review the logs for each of these failed runs to identify any specific errors or warnings.
2. **Environment Issues**: Verify if there are any known issues with the execution environm

In [17]:
# Question 3/6  [Tier 3 -- pipeline_sql_tool]
Q3 = "Show me borough_partition_load failure counts grouped by week over the last 90 days. Is it getting worse over time?"
print("=" * 70)
print(" Q3/6  [Tier 3 -- pipeline_sql_tool]")
print("=" * 70)
print(f" {Q3}")
print("-" * 70)
answer = ask(Q3)
print()
print(answer)


 Q3/6  [Tier 3 -- pipeline_sql_tool]
 Show me borough_partition_load failure counts grouped by week over the last 90 days. Is it getting worse over time?
----------------------------------------------------------------------
[TOOL] pipeline_sql_tool called with: SELECT TO_CHAR(execution_date, 'IW') AS week_number, COUNT(*) AS failure_count F

The failure counts for the `borough_partition_load` DAG grouped by week over the last 90 days are as follows:

- Week 5: 2 failures
- Week 6: 8 failures
- Week 7: 4 failures
- Week 8: 6 failures
- Week 9: 4 failures
- Week 10: 5 failures
- Week 11: 3 failures
- Week 12: 4 failures
- Week 13: 1 failure
- Week 14: 2 failures
- Week 15: 8 failures
- Week 16: 4 failures
- Week 17: 2 failures

From this data, we can observe that the number of failures has been fluctuating but generally increasing over time. Specifically:

- The highest failure count was in Weeks 6 and 15 with 8 failures each.
- There is a noticeable increase from Week 5 to Week 6, indi

In [18]:
# Question 4/6  [Tier 2 -- log_search_tool, silent issues]
Q4 = "Are the data quality check jobs running successfully, or are they also failing? Search the logs for schema drift check results."
print("=" * 70)
print(" Q4/6  [Tier 2 -- log_search_tool, silent issues]")
print("=" * 70)
print(f" {Q4}")
print("-" * 70)
answer = ask(Q4)
print()
print(answer)


 Q4/6  [Tier 2 -- log_search_tool, silent issues]
 Are the data quality check jobs running successfully, or are they also failing? Search the logs for schema drift check results.
----------------------------------------------------------------------
[TOOL] log_search_tool called with: schema drift check results

The `check_schema_drift` tasks within the `data_quality_checks` DAG are failing multiple times, as indicated by the following log entries:

- On March 19, 2026: `executor exited with code 1`
- On February 28, 2026: `executor exited with code 1`
- On February 12, 2026: `executor exited with code 1`
- On April 26, 2026: `executor exited with code 1`
- On April 19, 2026: `executor exited with code 1`

These failures suggest that there might be an issue related to the schema checks or the execution environment. To investigate further:

1. **Review Task Logs**: Check the logs for each of these failed runs to identify any specific errors or warnings.
2. **Schema Changes**: Verify if 

In [19]:
# Question 5/6  [Tier 3 -- pipeline_sql_tool, temporal pattern]
Q5 = "Break down average duration and failure rate for borough_partition_load by day of the week. Which days stand out as outliers? Use the the pipeline sql tool"
print("=" * 70)
print(" Q5/6  [Tier 3 -- pipeline_sql_tool, temporal pattern]")
print("=" * 70)
print(f" {Q5}")
print("-" * 70)
answer = ask(Q5)
print()
print(answer)


 Q5/6  [Tier 3 -- pipeline_sql_tool, temporal pattern]
 Break down average duration and failure rate for borough_partition_load by day of the week. Which days stand out as outliers? Use the the pipeline sql tool
----------------------------------------------------------------------
[TOOL] pipeline_sql_tool called with: SELECT TO_CHAR(execution_date, 'DY') AS day_of_week, AVG(duration_seconds) AS av
[TOOL] pipeline_sql_tool called with: SELECT TO_CHAR(execution_date, 'DY') AS day_of_week, AVG(duration_seconds) AS av

Based on the query results, here is a breakdown of the average duration and failure rate for the `borough_partition_load` DAG by day of the week over the last 90 days:

- **Friday**: Average Duration: 254.89 seconds, Failure Rate: 8.97%
- **Monday**: Average Duration: 351.19 seconds, Failure Rate: 12.5%
- **Saturday**: Average Duration: 245.58 seconds, Failure Rate: 11.54%
- **Sunday**: Average Duration: 253.47 seconds, Failure Rate: 8.97%
- **Thursday**: Average Duration: 

In [20]:
# Question 6/7  [Hybrid -- direct Oracle converged query, no agent]
# Calling hybrid_pipeline_search directly to demonstrate Oracle's converged
# query capability: a single SQL statement that combines a DAG filter with
# VECTOR_DISTANCE() ranking -- no application-level join, no separate vector store.
print("=" * 70)
print(" Q6/7  [Hybrid -- DAG filter + VECTOR_DISTANCE() in one Oracle SQL]")
print("=" * 70)
print(" borough_partition_load: semantic log search filtered to this DAG only")
print("-" * 70)
result = hybrid_pipeline_search.invoke({
    "query": "task exceeded runtime SLA source system slow",
    "dag_id": "borough_partition_load",
})
print(result)

 Q6/7  [Hybrid -- DAG filter + VECTOR_DISTANCE() in one Oracle SQL]
 borough_partition_load: semantic log search filtered to this DAG only
----------------------------------------------------------------------
[TOOL] hybrid_pipeline_search called | query=task exceeded runtime SLA source system slow | dag=borough_partition_load | from= | to=
[1] sim=0.2910 | ERROR | borough_partition_load.load_staten_island | 2026-04-22 08:19:30 | status=failed
    Task exceeded maximum runtime of 3600s -- Staten Island FHV source responded after 3614s (SLA: 3600s). Source system may be under maintenance. Last checkpoint: 2,602 of 2,063 rows committed.

[2] sim=0.2921 | ERROR | borough_partition_load.load_staten_island | 2026-03-02 08:19:30 | status=failed
    Task exceeded maximum runtime of 3600s -- Staten Island FHV source responded after 3614s (SLA: 3600s). Source system may be under maintenance. Last checkpoint: 2,430 of 1,606 rows committed.

[3] sim=0.2929 | ERROR | borough_partition_load.load_st

In [21]:
# Question 7/7  [Synthesis -- memory from Q1-Q6, no new queries]
Q7 = "Give me a full reliability report for the borough_partition_load DAG -- failures, patterns, recommendations."
print("=" * 70)
print(" Q7/7  [Synthesis -- memory from Q1-Q6, no new queries]")
print("=" * 70)
print(f" {Q7}")
print("-" * 70)
answer = ask(Q7)
print()
print(answer)

 Q7/7  [Synthesis -- memory from Q1-Q6, no new queries]
 Give me a full reliability report for the borough_partition_load DAG -- failures, patterns, recommendations.
----------------------------------------------------------------------

### Reliability Report for `borough_partition_load` DAG

#### Overview:
- **Total Failures in Last 90 Days**: 45
- **Average Duration by Day of Week**:
  - Friday: 254.89 seconds, Failure Rate: 8.97%
  - Monday: 351.19 seconds, Failure Rate: 12.5%
  - Saturday: 245.58 seconds, Failure Rate: 11.54%
  - Sunday: 253.47 seconds, Failure Rate: 8.97%
  - Thursday: 247.24 seconds, Failure Rate: 8.97%
  - Tuesday: 245.53 seconds, Failure Rate: 12.5%
  - Wednesday: 241.31 seconds, Failure Rate: 6.41%

#### Patterns and Observations:
- **High Failure Rates on Monday and Tuesday**: Both days have a failure rate of 12.5%, which is significantly higher than other days.
- **Increased Duration on Monday and Tuesday**: These days also show the highest average duration

## What Was Built

This notebook demonstrated a complete, locally-running pipeline intelligence agent
built on Oracle AI Database. Here is what each component contributed:

### Architecture

| Component | Role |
|-----------|------|
| Oracle AI Database | Single store for relational job metadata, free-text logs, and vector embeddings |
| `PIPELINE_JOBS` + `PIPELINE_LOGS` | Structured + unstructured data in the same schema |
| `PIPELINE_LOG_VECTORS` | Vector index on log text, queryable via `VECTOR_DISTANCE()` |
| `nomic-embed-text` | Purpose-built embedding model -- not the chat model |
| `qwen2.5:7b` (or your LLM) | Agent reasoning and natural language response generation |
| LangChain ReAct agent | Orchestration layer -- tool selection, multi-step reasoning |

### Key Capabilities Demonstrated

1. **Converged storage** -- vector embeddings and relational tables in one Oracle instance,
   no separate vector store required

2. **Hybrid query** -- SQL `WHERE` filters + `VECTOR_DISTANCE()` ranking in a single
   statement (`hybrid_pipeline_search` tool)

3. **Dedicated embedding model** -- `nomic-embed-text` produces semantically precise
   vectors for log similarity search, separate from the reasoning LLM

4. **Agent memory** -- chat history is preserved across questions, so Q7 can reference
   findings from Q1-Q6 without re-running any queries

5. **Tier 2 discovery** -- schema drift, silent data quality issues, and deduplication
   events that have no structured error code were surfaced purely by vector search

6. **Portability** -- swap Ollama for OpenAI by changing three variables in the
   config cell; all LangChain code is identical

### Resources

- [Oracle AI Database 26ai Free Tier](https://www.oracle.com/database/free/)
- [Oracle Developer Resources](https://www.oracle.com/developer/resources/)
- [langchain-oracledb Documentation](https://python.langchain.com/docs/integrations/vectorstores/oracle/)
- [Oracle AI Developer Hub](https://github.com/oracle-devrel/oracle-ai-developer-hub)

---
2026 Andreas Kretz LearnDataEngineering.com